[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/01_agente_base/01_proyecto_agente_base.ipynb)

# Proyecto 01 · Extiende SupportOps v0 con una herramienta nueva y reproduce 3 fallos

| | |
|---|---|
| **Nivel** | 🟢 Básico |
| **Duración estimada** | ≈ 1 h 50 min |
| **Servicios de Compose** | Ninguno (todo simulado) |
| **RAM aproximada** | < 300 MB |
| **Prerrequisitos** | Lección `01_agente_base` (bucle del agente, herramientas, mock, fallos). Módulo 00 (proyecto: `check_stack`) |
| **Conexión** | Parte de **SupportOps v0** y te deja un agente con `check_stock` y un detector de síntomas que usarás como punto de partida en el módulo 03 (instrumentación) |

## Contexto de negocio

Eres ingeniero/a de IA en **SupportOps**. Marketing quiere que el agente conteste «¿hay stock del teclado K87?» sin escalar a una persona.
Antes de lanzarlo, Calidad te pide dos cosas: (1) **la herramienta nueva sin romper nada** (el resto de tickets debe comportarse exactamente igual) y (2) **un catálogo de síntomas**: para tres fallos que ya conocéis (bucle de herramientas, alucinación, timeout), qué «huella» dejan en el resultado del agente, y un detector que *solo mire el resultado* (sin conocer la verdad) para ver cuánto se puede detectar sin observabilidad especializada.

## Datos

Tráfico sintético de SupportOps (semilla fija `42`, esquema en `schema.TICKET_SCHEMA`) generado con `generate_tickets`; no hay descargas. **Este proyecto no usa servicios**, así que no hay modo degradado: funciona siempre.

## Entregables y criterios de evaluación (rúbrica)

| # | Entregable | Criterio medido | Umbral (calibrado con la solución en `mock`, semilla 42) |
|---|---|---|---|
| R1 | `check_stock(sku)` | Coincide con `catalog.stock()` para los 10 SKU y gestiona un SKU desconocido sin excepción | 10/10 SKU + 1 caso desconocido |
| R2 | Herramienta registrada | Esquema JSON válido: `sku` en `properties` y en `required` | cumple |
| R3 | El agente la usa | De 8 preguntas de stock: la usa **exactamente una vez** y el número que cita coincide con el stock real | ≥ 7/8 |
| R4 | Sin regresiones | En 150 tickets del tráfico normal, `tools_used` idéntico al agente sin la regla nueva | 100 % idénticos |
| R5 | Tres huellas reproducibles | Tus `SIGNATURES` son verdaderas para su fallo en los 20 tickets de prueba y **falsas** en los 20 tickets sanos | 3/3 fallos con 100 % sensibilidad y 0 falsos positivos |
| R6 | Detector ciego | `detect(result)` (sin leer `injected_faults`/`grounded`/`pii_leaked`) sobre 400 tickets con fallos: precisión global ≥ 0,90 y recall de `tool_loop`, `tool_timeout`, `tool_error`, `pii_leak` ≥ 0,90 | ver celda de la rúbrica |

> 📏 Los umbrales salen de lo que mide la solución de referencia en `mock` con semilla fija; no dependen de la velocidad de tu máquina porque el tiempo es **simulado**. La solución obtiene: R3 = 8/8, R4 = 150/150, R5 = 3/3 con 0 falsos positivos y,
> en R6 sobre 400 tickets, precisión 1,00 y recall 1,00 en `tool_loop` (2 casos), `tool_timeout` (5), `tool_error` (14) y `pii_leak` (6); el umbral de 0,90 deja margen y los conteos son pequeños (no extraigas conclusiones estadísticas de ellos).

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz]"], check=True)
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import json, re
from supportops import catalog
from supportops.llm.mock import ORDER_RE, SKU_RE
_ = check_stack([])      # este proyecto no necesita servicios

## Paso 1 · La herramienta `check_stock`

**TODO 1.** Implementa `check_stock(sku: str) -> dict` que devuelva `{"found": True, "sku": ..., "name": ..., "units": int, "available": bool}` usando `catalog.stock()` (diccionario `sku → unidades`) y `catalog.PRODUCTS` (lista de `(sku, nombre, precio)`).
Para un SKU desconocido devuelve `{"found": False, "sku": sku}` (**no** lances una excepción: es un resultado de negocio, no un fallo de infraestructura).

<details>
<summary>🪜 Pista 1</summary>

Construye un diccionario `nombres = {sku: nombre for sku, nombre, _ in catalog.PRODUCTS}` y consulta `catalog.stock().get(sku)`. Ten cuidado: `0` unidades es un valor válido (no uses `if not units`).

</details>

In [ ]:
def check_stock(sku: str) -> dict:
    # TODO 1
    raise NotImplementedError("TODO 1")

try:
    print(check_stock("SKU-1001"), check_stock("SKU-9999"))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 2 · Registrarla con su esquema

**TODO 2.** Crea un `ToolRegistry` con las herramientas por defecto (`default_tools()`) y **registra** `check_stock` como `Tool(name, description, parameters, fn, latency_median_s)`.
`parameters` es un JSON Schema de objeto con la propiedad `sku` (string) **obligatoria**. Usa `latency_median_s=0.06`.

<details>
<summary>🪜 Pista 1</summary>

`reg = default_tools()` y `reg.register(Tool(name="check_stock", description="...", parameters={...}, fn=check_stock, latency_median_s=0.06))`. Revisa cómo se ve el esquema de `issue_refund` con `reg.get('issue_refund').schema()`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Esquema: `{"type": "object", "properties": {"sku": {"type": "string", "description": "..."}}, "required": ["sku"]}`.

</details>

In [ ]:
def build_registry() -> ToolRegistry:
    # TODO 2: devuelve un registro con las herramientas por defecto + check_stock
    raise NotImplementedError("TODO 2")

try:
    REG = build_registry(); print(REG.names())
except NotImplementedError as e:
    REG = None; print("Pendiente:", e)

## Paso 3 · Enseñar al agente (mock) cuándo usarla

El mock decide con **reglas** (`Rule`): palabras clave → pasos de herramienta → plantilla de respuesta. Observa las reglas por defecto con `default_rules()` y añade la tuya.

**TODO 3a.** Implementa `sku_from_text(text)`: devuelve el SKU mencionado (`SKU-####`, usa `SKU_RE`) **o** el SKU del producto cuyo nombre aparece en el texto (ignora mayúsculas; compara con `catalog.PRODUCTS`, p. ej. «teclado mecánico K87» → `SKU-1001`; basta con que aparezca el nombre completo o su última palabra, p. ej. «K87», «M3»). Si no hay, `""`.

**TODO 3b.** Implementa `build_llm()` que devuelva un `MockLLM(seed=SEED)` con una `Rule("stock", keywords=(...), steps=[ToolStep("check_stock", args=...)], answer=..., needs=..., missing_answer=..., clarify=True, priority=25)`.
La respuesta debe decir cuántas unidades quedan (o que no hay stock / que el SKU no existe). Palabras clave sugeridas: `"stock"`, `"disponib"`, `"quedan"`, `"unidades"`.

<details>
<summary>🪜 Pista 1</summary>

Mira `default_rules()` en `supportops/llm/mock.py` (puedes imprimirlo con `inspect.getsource`). Una `Rule` necesita: `intent`, `keywords` (tupla de *subcadenas* en minúsculas), `steps` (lista de `ToolStep(tool, args)` donde `args(text, results)` devuelve el dict de argumentos), `answer(text, results)` (devuelve el texto final; `results['check_stock']` es el resultado de tu herramienta), y opcionalmente `needs(text)` + `missing_answer` + `clarify=True` para pedir el dato que falta.

</details>

<details>
<summary>🪜 Pista 2</summary>

`ToolStep("check_stock", lambda t, r: {"sku": sku_from_text(t)})`. `answer=lambda t, r: ...` con tres casos: `not r['check_stock']['found']`, `units == 0`, y el caso normal («Quedan {units} unidades de {name}»). `priority=25` hace que gane a las reglas por defecto cuando el ticket habla de stock.

</details>

In [ ]:
def sku_from_text(text: str) -> str:
    # TODO 3a
    raise NotImplementedError("TODO 3a")

def build_llm() -> MockLLM:
    # TODO 3b
    raise NotImplementedError("TODO 3b")

try:
    print(sku_from_text("¿Hay stock del teclado mecánico K87?"), sku_from_text("¿Quedan unidades del SKU-1005?"), repr(sku_from_text("hola")))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 4 · Prueba funcional y *test de regresión*

Con las piezas anteriores montamos el agente nuevo. Dos pruebas: (a) **8 preguntas de stock** y (b) **regresión**: los 150 tickets normales deben usar *exactamente las mismas herramientas* que con el agente original (una regla nueva no puede «robar» tickets de otras intenciones).

**TODO 4.** Implementa `build_agent()` que devuelva `SupportAgent(llm=build_llm(), tools=build_registry())`. Después ejecuta la celda de pruebas (ya escrita).

In [ ]:
def build_agent() -> SupportAgent:
    # TODO 4
    raise NotImplementedError("TODO 4")

STOCK_Q = ["¿Hay stock del teclado mecánico K87?", "¿Quedan unidades del SKU-1005?", "¿Está disponible el ratón inalámbrico M3?", "¿Tenéis stock del SKU-1010?",
           "¿Cuántas unidades quedan de la webcam 4K?", "¿Hay disponibilidad del SSD NVMe 1 TB?", "¿Queda stock de los auriculares ANC H5?", "¿Hay stock del SKU-1003?"]

def run_stock_tests(agent) -> list[dict]:
    out = []
    for i, q in enumerate(STOCK_Q):
        r = agent.run(Ticket(f"T-stock{i}", "C-0001", "standard", "chat", q, "stock", SIM_EPOCH_S))
        sku = sku_from_text(q); real = catalog.stock().get(sku)
        out.append({"pregunta": q, "tools": r.tools_used, "units_reales": real, "usa_1_vez": r.tools_used == ["check_stock"],
                    "cita_numero": bool(re.search(rf"\b{real}\b", r.answer)) if real else ("no hay stock" in r.answer.lower() or "no encuentro" in r.answer.lower()), "respuesta": r.answer[:70]})
    return out

try:
    NEW_AGENT = build_agent()
    ST = pd.DataFrame(run_stock_tests(NEW_AGENT)); display(ST)
except NotImplementedError as e:
    NEW_AGENT = None; print("Pendiente:", e)

## Paso 5 · Reproducir tres fallos y definir su «huella»

Ahora la parte de observabilidad. Para cada fallo (**`tool_loop`**, **`hallucination`**, **`tool_timeout`**) sobre una pregunta de stock:

1. **TODO 5a.** `inject(kind, text)` devuelve un `Ticket` con ese fallo (`FaultSpec(kind)`), y ejecútalo con tu agente.
2. Mira la tabla de resultados que imprime la celda siguiente (estado, pasos, latencia, tokens, `grounded`…) y **escribe tus `SIGNATURES`**: una función `lambda r: bool` por fallo basada **solo en lo observable**
   (`r.status`, `r.n_tool_calls`, `r.latency_s`, `r.steps`…). Debe ser verdadera para su fallo y **falsa** para tickets sanos (sin falsos positivos).

**TODO 5b.** Completa el diccionario `SIGNATURES`.

<details>
<summary>🪜 Pista 1</summary>

Tabla de huellas esperada (compruébala con tus datos): `tool_loop` → estado `max_steps` y tantas herramientas como `max_steps`; `hallucination` → **0 herramientas** en una pregunta que necesita una; `tool_timeout` → algún paso `tool` con `error_type == 'timeout'` (o latencia > 5 s).

</details>

<details>
<summary>🪜 Pista 2</summary>

Ejemplos de predicados: `lambda r: r.status == 'max_steps'`; `lambda r: r.n_tool_calls == 0 and r.status == 'ok'`; `lambda r: any(s.kind == 'tool' and s.error_type == 'timeout' for s in r.steps)`. Cuidado: «0 herramientas» también ocurre en la pregunta de aclaración (sin SKU): usa solo preguntas con SKU para esta prueba.

</details>

In [ ]:
def inject(kind: str, text: str, i: int = 0) -> Ticket:
    # TODO 5a: ticket con el fallo `kind` (FaultSpec(kind)) y el texto dado
    raise NotImplementedError("TODO 5a")

SIGNATURES = {
    # TODO 5b: predicado r -> bool por fallo, basado solo en lo observable
    "tool_loop": None,
    "hallucination": None,
    "tool_timeout": None,
}

try:
    rows = []
    for kind in ["tool_loop", "hallucination", "tool_timeout"]:
        r = NEW_AGENT.run(inject(kind, "¿Hay stock del teclado mecánico K87?"))
        rows.append({"fallo": kind, "estado": r.status, "pasos": len(r.steps), "herramientas": r.n_tool_calls, "latencia_s": round(r.latency_s, 2), "tokens_in": r.input_tokens,
                     "errores_de_paso": [s.error_type for s in r.steps if s.error_type], "fundamentada": r.grounded})
    display(pd.DataFrame(rows).set_index("fallo"))
except (NotImplementedError, AttributeError, TypeError) as e:
    print("Pendiente (completa los TODO anteriores):", type(e).__name__, e)

## Paso 6 · Un detector ciego

**TODO 6.** Implementa `detect(r) -> set[str]`: dado **solo** el `AgentResult` (sin `injected_faults`, `grounded` ni `pii_leaked`, que son la verdad de terreno), devuelve el conjunto de fallos que sospechas de entre
`{"tool_loop", "tool_timeout", "tool_error", "llm_error", "pii_leak", "hallucination", "latency_spike"}`.
Pistas de dónde mirar: el **estado**, los **pasos con error** (`s.error_type`), el **número de herramientas**, la **latencia** y el **texto de la respuesta** (¿contiene un correo o una tarjeta?).
Luego evalúa precisión y recall frente a la verdad de terreno en 400 tickets con fallos de fondo.

<details>
<summary>🪜 Pista 1</summary>

Reglas sencillas: `status=='max_steps'` → `tool_loop`; paso `tool` con `error_type=='timeout'` → `tool_timeout`; otro `error_type` en paso `tool` → `tool_error`; `status=='error'` → `llm_error`; regex de correo (`[\\w.]+@[\\w.]+`) o de tarjeta (`\\d{4} \\d{4} \\d{4} \\d{4}`) en la respuesta → `pii_leak`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`hallucination`: la respuesta afirma datos concretos (un número de pedido `ORD-`, un seguimiento `TRK`, «aprobado», «unidades») pero `n_tool_calls == 0`. `latency_spike`: latencia por encima de un umbral robusto (p. ej. mediana de sanos × 3), pero ojo con los falsos positivos de la cola natural.

</details>

In [ ]:
import re

def detect(r) -> set[str]:
    # TODO 6
    raise NotImplementedError("TODO 6")

def evaluate_detector(detect_fn, n=400):
    tk = generate_tickets(n, seed=SEED, fault_rates=DEFAULT_FAULT_RATES)
    res = run_tickets(NEW_AGENT or SupportAgent(), tk)
    kinds = list(FAULT_KINDS)
    tp = dict.fromkeys(kinds, 0); fp = dict.fromkeys(kinds, 0); fn = dict.fromkeys(kinds, 0)
    for r in res:
        truth, pred = set(r.injected_faults), detect_fn(r)
        for k in kinds:
            tp[k] += (k in truth) and (k in pred); fp[k] += (k not in truth) and (k in pred); fn[k] += (k in truth) and (k not in pred)
    T = pd.DataFrame({"verdad": {k: tp[k] + fn[k] for k in kinds}, "TP": tp, "FP": fp, "FN": fn})
    T["recall"] = (T.TP / T.verdad.replace(0, np.nan)).round(2); T["precisión"] = (T.TP / (T.TP + T.FP).replace(0, np.nan)).round(2)
    return T, (sum(tp.values()) / max(1, sum(tp.values()) + sum(fp.values())))

try:
    TABLE, PREC = evaluate_detector(detect)
    display(TABLE); print(f"precisión global = {PREC:.2f}")
except NotImplementedError as e:
    TABLE, PREC = None, None; print("Pendiente:", e)

## Paso 7 · Rúbrica

Ejecuta la celda para puntuar tu trabajo. Los umbrales provienen de la solución de referencia.

In [ ]:
def run_rubric():
    rows = []
    def add(r, desc, ok, detail=""): rows.append({"criterio": r, "descripción": desc, "resultado": "✅ OK" if ok else "❌ FALLA", "detalle": detail})
    # R1
    try:
        ok1 = all((lambda d, s: d.get("found") and d["units"] == s and d["available"] == (s > 0))(check_stock(k), v) for k, v in catalog.stock().items()) and check_stock("SKU-9999") == {"found": False, "sku": "SKU-9999"}
        add("R1", "check_stock correcta (10 SKU + desconocido)", ok1)
    except Exception as e: add("R1", "check_stock correcta", False, repr(e)[:80])
    # R2
    try:
        sch = REG.get("check_stock").schema()["parameters"]
        add("R2", "esquema válido (sku requerido)", "sku" in sch["properties"] and "sku" in sch.get("required", []))
    except Exception as e: add("R2", "esquema válido", False, repr(e)[:80])
    # R3
    try:
        ST = pd.DataFrame(run_stock_tests(NEW_AGENT)); n = int((ST.usa_1_vez & ST.cita_numero).sum())
        add("R3", "≥ 7/8 preguntas de stock bien resueltas", n >= 7, f"{n}/8")
    except Exception as e: add("R3", "preguntas de stock", False, repr(e)[:80])
    # R4
    try:
        tk = generate_tickets(150, seed=SEED); base = run_tickets(SupportAgent(), tk); new = run_tickets(NEW_AGENT, tk)
        same = sum(a.tools_used == b.tools_used for a, b in zip(base, new))
        add("R4", "sin regresiones (100 % idénticos)", same == len(tk), f"{same}/{len(tk)}")
    except Exception as e: add("R4", "regresión", False, repr(e)[:80])
    # R5
    try:
        det = {}
        for kind, pred in SIGNATURES.items():
            sick = [NEW_AGENT.run(inject(kind, STOCK_Q[i % 8] if i % 8 != 1 else "¿Hay stock del SKU-1003?", i)) for i in range(20)]
            healthy = [NEW_AGENT.run(Ticket(f"T-h{i}", "C-0001", "standard", "chat", STOCK_Q[i % 8], "stock", SIM_EPOCH_S + i)) for i in range(20)]
            det[kind] = (sum(map(pred, sick)) / 20, sum(map(pred, healthy)))
        add("R5", "huellas: sensibilidad 100 % y 0 falsos positivos", all(s == 1.0 and f == 0 for s, f in det.values()), str(det))
    except Exception as e: add("R5", "huellas", False, repr(e)[:80])
    # R6
    try:
        T, prec = evaluate_detector(detect)
        rec = {k: float(T.loc[k, "recall"]) for k in ["tool_loop", "tool_timeout", "tool_error", "pii_leak"]}
        add("R6", "detector: precisión ≥ 0,90 y recall ≥ 0,90", prec >= 0.90 and all(v >= 0.90 for v in rec.values()), f"precisión={prec:.2f} recall={rec}")
    except Exception as e: add("R6", "detector ciego", False, repr(e)[:80])
    return pd.DataFrame(rows).set_index("criterio")

display(run_rubric())

---
## Retos extra (nivel experto)

1. **Guarda de bucles en el agente:** crea `class GuardedAgent(SupportAgent)` que detecte la *misma herramienta con los mismos argumentos* repetida ≥ 3 veces y corte con una respuesta de escalado. Mide el ahorro de tokens/coste en el tráfico con `tool_loop` y los falsos positivos en tráfico sano.
2. **Idempotencia del reembolso:** protege `issue_refund` con una *clave de idempotencia* (`order_id`) para que un reintento tras un timeout no duplique el reembolso. Demuéstralo con `tool_timeout` + reintento.
3. **Presupuesto de pasos por intención:** calcula con 1000 tickets el percentil 99 de pasos por intención y propón un `max_steps` distinto por intención. ¿Cuántos falsos cortes provoca?
4. **Cambia de proveedor:** con una clave propia (solo por variables de entorno), ejecuta las 8 preguntas de stock con `LLM_PROVIDER=anthropic|openai|ollama`. ¿Qué diferencias de pasos, tokens y latencia ves frente al mock? (No es parte de la rúbrica.)

## Reflexión (producción e hilo conductor)

- ⚠️ **Validez:** en el mock la alucinación usa frases fijas, por eso un *regex* la detecta al 100 %. Con un LLM real no existe esa regularidad: una alucinación solo se detecta comparando la respuesta con las fuentes (evaluación online, módulo 14). ¿Qué parte de tu detector **sí** generalizaría a producción (pista: PII por patrones, bucles por repetición, errores por estado de los pasos) y cuál no?
- Tu detector ciego, ¿qué fallos **no puede** detectar con datos reales y qué dato adicional necesitaría?
- Si el 100 % de tus herramientas fallara a la vez, ¿cuántos de los *síntomas* que has definido se activarían? ¿Cuál sería la primera alerta útil?
- Has añadido una herramienta y un test de regresión: ¿qué parte de la telemetría del módulo 03 te habría avisado *en producción* de que la regla nueva robaba tickets?
- En el módulo 03 instrumentarás este mismo agente. ¿Qué atributos pondrías en el span de `check_stock` para depurar discrepancias de inventario?

---
# ⛔ SPOILER — intenta resolverlo primero

A continuación, la **solución de referencia** completa y ejecutable.

In [ ]:
# ===================== SOLUCIÓN DE REFERENCIA =====================
NAMES = {sku: nombre for sku, nombre, _ in catalog.PRODUCTS}

# TODO 1
def check_stock(sku: str) -> dict:
    units = catalog.stock().get(sku)
    if units is None:
        return {"found": False, "sku": sku}
    return {"found": True, "sku": sku, "name": NAMES[sku], "units": units, "available": units > 0}

# TODO 2
def build_registry() -> ToolRegistry:
    reg = default_tools()
    reg.register(Tool(name="check_stock", description="Consulta las unidades en stock de un producto por su SKU (SKU-####).",
                      parameters={"type": "object", "properties": {"sku": {"type": "string", "description": "SKU del producto"}}, "required": ["sku"]},
                      fn=check_stock, latency_median_s=0.06))
    return reg

# TODO 3a
def sku_from_text(text: str) -> str:
    m = SKU_RE.search(text)
    if m:
        return m.group(0)
    low = text.lower()
    for sku, nombre, _ in catalog.PRODUCTS:
        if nombre.lower() in low or nombre.lower().split()[-1] in low.replace("?", " ").replace(",", " ").split():
            return sku
    return ""

# TODO 3b
def build_llm() -> MockLLM:
    def answer(t, r):
        s = r["check_stock"]
        if not s.get("found"):
            return f"No encuentro el producto {s['sku']} en el catálogo."
        if s["units"] == 0:
            return f"Ahora mismo no hay stock de {s['name']} ({s['sku']}). ¿Quiere que le avisemos cuando vuelva a estar disponible?"
        return f"Quedan {s['units']} unidades de {s['name']} ({s['sku']})."
    llm = MockLLM(seed=SEED)
    llm.add_rule(Rule("stock", keywords=("stock", "disponib", "quedan", "unidades"),
                      steps=[ToolStep("check_stock", lambda t, r: {"sku": sku_from_text(t)})], answer=answer,
                      needs=lambda t: bool(sku_from_text(t)), missing_answer="¿De qué producto quiere consultar el stock? Indíqueme su nombre o su SKU.",
                      clarify=True, priority=25))
    return llm

# TODO 4
def build_agent() -> SupportAgent:
    return SupportAgent(llm=build_llm(), tools=build_registry())

# TODO 5a / 5b
def inject(kind: str, text: str, i: int = 0) -> Ticket:
    return Ticket(f"T-{kind}-{i}", "C-0001", "standard", "chat", text, "stock", SIM_EPOCH_S + i, faults=[FaultSpec(kind)])

SIGNATURES = {
    "tool_loop": lambda r: r.status == "max_steps" and r.n_tool_calls >= 5,
    "hallucination": lambda r: r.status == "ok" and r.n_tool_calls == 0 and r.n_llm_calls == 1,
    "tool_timeout": lambda r: any(s.kind == "tool" and s.error_type == "timeout" for s in r.steps),
}

# TODO 6: detector ciego
EMAIL, CARD = re.compile(r"[\w.+-]+@[\w-]+\.\w+"), re.compile(r"\b\d{4} \d{4} \d{4} \d{4}\b")
CLAIMS = re.compile(r"TRK\d+|aprobado|unidades|garantía|días")   # afirmaciones concretas (sin «ORD-»: lo usa también la pregunta de aclaración)

def detect(r) -> set[str]:
    found: set[str] = set()
    errs = [s.error_type for s in r.steps if s.kind == "tool" and s.error_type]
    if r.status == "max_steps": found.add("tool_loop")
    if "timeout" in errs: found.add("tool_timeout")
    if any(e and e != "timeout" for e in errs): found.add("tool_error")
    if any(s.kind == "llm" and s.status == "error" for s in r.steps): found.add("llm_error")   # también los transitorios (el reintento los oculta en el estado)
    if EMAIL.search(r.answer) or CARD.search(r.answer): found.add("pii_leak")
    if r.status == "ok" and r.n_tool_calls == 0 and CLAIMS.search(r.answer): found.add("hallucination")
    if r.latency_s > 8 and not (found & {"tool_loop", "tool_timeout", "llm_error"}): found.add("latency_spike")
    return found

NEW_AGENT = build_agent()
REG = build_registry()
display(pd.DataFrame(run_stock_tests(NEW_AGENT)))
TABLE, PREC = evaluate_detector(detect); display(TABLE); print(f"precisión global = {PREC:.2f}")
display(run_rubric())